In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("job_market_unemployment_trends.csv")

# Display shape
print("Shape:", df.shape)

# Display variable types
print("\nVariable types:")
print(df.dtypes)

# Display first rows
print("\nFirst 5 rows:")
print(df.head())

Shape: (1000, 8)

Variable types:
id                             int64
date                          object
location                      object
unemployment_rate            float64
job_postings                   int64
in_demand_skills              object
average_age                    int64
college_degree_percentage      int64
dtype: object

First 5 rows:
   id        date      location  unemployment_rate  job_postings  \
0   1  2023-10-07       Houston                6.8          4894   
1   2  2025-05-24    Washington               11.1          2695   
2   3  2024-09-28       Chicago                7.3          1174   
3   4  2024-12-28  Indianapolis               11.2          3708   
4   5  2023-09-10      New York               13.7           268   

                                    in_demand_skills  average_age  \
0            Agile Methodologies, Project Management           44   
1     Data Analysis, UX/UI Design, Digital Marketing           50   
2  Agile Methodologies, D

In [3]:
# Count missing values per column
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
id                           0
date                         0
location                     0
unemployment_rate            0
job_postings                 0
in_demand_skills             0
average_age                  0
college_degree_percentage    0
dtype: int64


In [4]:
# Select numerical columns relevant to the business question
numerical_cols = ["unemployment_rate", "job_postings", "average_age", "college_degree_percentage"]

# Detect outliers using IQR method
print("=== Outlier Detection (IQR Method) ===\n")

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)               # First quartile
    Q3 = df[col].quantile(0.75)               # Third quartile
    IQR = Q3 - Q1                             # Interquartile range
    lower_bound = Q1 - 1.5 * IQR             # Lower bound
    upper_bound = Q3 + 1.5 * IQR             # Upper bound

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]  # Filter outliers

    print(f"[{col}]")
    print(f"  Q1={Q1:.2f} | Q3={Q3:.2f} | IQR={IQR:.2f}")
    print(f"  Bounds: [{lower_bound:.2f} ; {upper_bound:.2f}]")
    print(f"  Outliers detected: {len(outliers)}\n")

=== Outlier Detection (IQR Method) ===

[unemployment_rate]
  Q1=5.40 | Q3=11.80 | IQR=6.40
  Bounds: [-4.20 ; 21.40]
  Outliers detected: 0

[job_postings]
  Q1=1212.75 | Q3=3779.00 | IQR=2566.25
  Bounds: [-2636.62 ; 7628.38]
  Outliers detected: 0

[average_age]
  Q1=31.00 | Q3=44.00 | IQR=13.00
  Bounds: [11.50 ; 63.50]
  Outliers detected: 0

[college_degree_percentage]
  Q1=46.00 | Q3=75.00 | IQR=29.00
  Bounds: [2.50 ; 118.50]
  Outliers detected: 0



In [6]:
# No missing values and no outliers detected — dataset is clean
print("Dataset is clean. No correction needed.")

Dataset is clean. No correction needed.


In [7]:
# Check current types before any conversion
print("Current types:")
print(df.dtypes)

Current types:
id                             int64
date                          object
location                      object
unemployment_rate            float64
job_postings                   int64
in_demand_skills              object
average_age                    int64
college_degree_percentage      int64
dtype: object


In [8]:
# Convert date column to datetime format for SQL and Power BI compatibility
df["date"] = pd.to_datetime(df["date"])

# Verify conversion
print("Date type after conversion:", df["date"].dtype)
print(df["date"].head())

Date type after conversion: datetime64[ns]
0   2023-10-07
1   2025-05-24
2   2024-09-28
3   2024-12-28
4   2023-09-10
Name: date, dtype: datetime64[ns]


In [9]:
# Normalize location: strip spaces and apply title case for SQL/Power BI consistency
df["location"] = df["location"].str.strip().str.title()

# Verify unique values
print("Unique locations:", df["location"].nunique())
print(df["location"].unique())

Unique locations: 20
['Houston' 'Washington' 'Chicago' 'Indianapolis' 'New York' 'Los Angeles'
 'Phoenix' 'San Jose' 'Austin' 'Dallas' 'San Francisco' 'Columbus'
 'San Antonio' 'Jacksonville' 'Seattle' 'Charlotte' 'Philadelphia'
 'Denver' 'San Diego' 'Fort Worth']


In [10]:
# Normalize skills: strip extra spaces and apply title case for consistent segmentation
df["in_demand_skills"] = df["in_demand_skills"].str.strip().str.title()

# Verify unique values
print("Unique skills:", df["in_demand_skills"].nunique())
print(df["in_demand_skills"].unique())

Unique skills: 859
['Agile Methodologies, Project Management'
 'Data Analysis, Ux/Ui Design, Digital Marketing'
 'Agile Methodologies, Digital Marketing, Machine Learning, Cloud Computing, Cybersecurity'
 'Cloud Computing, Ux/Ui Design'
 'Sql, Machine Learning, Python, Project Management, Customer Service'
 'Cloud Computing, Ux/Ui Design, Project Management'
 'Data Analysis, Project Management'
 'Customer Service, Ux/Ui Design, Cybersecurity, Python, Sql'
 'Digital Marketing, Javascript' 'Cloud Computing, Javascript'
 'Data Analysis, Cloud Computing'
 'Project Management, Digital Marketing, Agile Methodologies'
 'Agile Methodologies, Javascript, Ux/Ui Design'
 'Cybersecurity, Data Analysis' 'Project Management, Sales, Ux/Ui Design'
 'Sales, Ux/Ui Design, Javascript, Digital Marketing'
 'Sql, Javascript, Customer Service, Ux/Ui Design'
 'Cybersecurity, Javascript, Sql, Cloud Computing' 'Javascript, Python'
 'Digital Marketing, Customer Service, Cybersecurity, Javascript, Agile Methodolo

In [11]:
# Check that college_degree_percentage is between 0 and 100
out_of_range = df[(df["college_degree_percentage"] < 0) | (df["college_degree_percentage"] > 100)]

print("Values out of range [0-100]:", len(out_of_range))
print(df["college_degree_percentage"].describe())

Values out of range [0-100]: 0
count    1000.000000
mean       60.608000
std        17.404173
min        30.000000
25%        46.000000
50%        60.000000
75%        75.000000
max        90.000000
Name: college_degree_percentage, dtype: float64


In [12]:
# Final check: all types must be compatible with SQL and Power BI
print("\nFinal types after conversion:")
print(df.dtypes)

# Display cleaned sample
print("\nCleaned dataset sample:")
print(df.head())


Final types after conversion:
id                                    int64
date                         datetime64[ns]
location                             object
unemployment_rate                   float64
job_postings                          int64
in_demand_skills                     object
average_age                           int64
college_degree_percentage             int64
dtype: object

Cleaned dataset sample:
   id       date      location  unemployment_rate  job_postings  \
0   1 2023-10-07       Houston                6.8          4894   
1   2 2025-05-24    Washington               11.1          2695   
2   3 2024-09-28       Chicago                7.3          1174   
3   4 2024-12-28  Indianapolis               11.2          3708   
4   5 2023-09-10      New York               13.7           268   

                                    in_demand_skills  average_age  \
0            Agile Methodologies, Project Management           44   
1     Data Analysis, Ux/Ui Design, Di

In [13]:
# Rename columns to business-friendly names for Power BI and SQL readability
df = df.rename(columns={
    "location"                 : "region",
    "job_postings"             : "job_offers",
    "in_demand_skills"         : "skills_required",
    "college_degree_percentage": "graduation_rate"
})

In [14]:
# Verify all columns have been renamed correctly
print("Columns after renaming:")
print(df.columns.tolist())

Columns after renaming:
['id', 'date', 'region', 'unemployment_rate', 'job_offers', 'skills_required', 'average_age', 'graduation_rate']


In [16]:
# Extract year from date column for annual trend analysis
df["year"] = df["date"].dt.year

# Extract month from date column for seasonality analysis
df["month"] = df["date"].dt.month

In [17]:
# Categorize unemployment rate into severity levels for KPI cards
df["unemployment_level"] = pd.cut(
    df["unemployment_rate"],
    bins=[0, 5, 10, 100],
    labels=["Low", "Moderate", "High"]
)

In [18]:
# Compute market tension index: ratio of job offers to unemployment pressure
df["market_tension"] = df["job_offers"] / (df["unemployment_rate"] + 1)

# Round to 2 decimals for readability
df["market_tension"] = df["market_tension"].round(2)

In [19]:
# Segment average age into generational brackets for demographic analysis
df["age_range"] = pd.cut(
    df["average_age"],
    bins=[0, 30, 45, 100],
    labels=["Young", "Active", "Senior"]
)

In [20]:
# Categorize diploma rate into education levels for territorial comparison
df["education_level"] = pd.cut(
    df["graduation_rate"],
    bins=[0, 30, 60, 100],
    labels=["Low", "Medium", "High"]
)

In [24]:
# Segment job postings volume into market dynamism categories
df["categories_offers"] = pd.cut(
    df["job_offers"],
    bins=[0, 500, 1500, 999999],
    labels=["Weak", "Medium", "Strong"]
)

In [25]:
# Group skills into macro sectors based on keywords
def categorize_skill(skill):
    skill = str(skill).lower()
    if any(k in skill for k in ["python", "sql", "data", "machine", "cloud", "cyber", "developer", "software"]):
        return "Tech & Data"
    elif any(k in skill for k in ["marketing", "seo", "communication", "content", "social"]):
        return "Marketing & Com"
    elif any(k in skill for k in ["finance", "accounting", "audit", "budget"]):
        return "Finance & Management"
    elif any(k in skill for k in ["nurse", "health", "medical", "care"]):
        return "Health & Social"
    elif any(k in skill for k in ["sales", "commercial", "retail", "crm"]):
        return "Trade & Sales"
    else:
        return "Others"

# Apply skill categorization to each row
df["family_skills"] = df["skills_required"].apply(categorize_skill)

In [26]:
# Display new columns created
new_cols = ["year", "month", "unemployment_level", "market_tension",
            "age_range", "education_level", "categories_offers", "family_skills"]

print("New variables preview:")
print(df[new_cols].head(10))

# Check data types of new variables
print("\nNew variable types:")
print(df[new_cols].dtypes)

New variables preview:
   year  month unemployment_level  market_tension age_range education_level  \
0  2023     10           Moderate          627.44    Active            High   
1  2025      5               High          222.73    Senior            High   
2  2024      9           Moderate          141.45    Senior          Medium   
3  2024     12               High          303.93    Active            High   
4  2023      9               High           18.23    Active            High   
5  2024     12                Low          297.50    Active          Medium   
6  2023      9               High          189.39    Active          Medium   
7  2024     10               High          448.74    Active          Medium   
8  2024      8           Moderate          245.30     Young            High   
9  2024      2               High           60.30    Active          Medium   

  categories_offers    family_skills  
0            Strong           Others  
1            Strong      Tech

In [28]:
# Drop columns with no analytical or decisional value
df = df.drop(columns=["id", "date", "average_age", "graduation_rate"])

In [29]:
# Verify remaining columns after drop
print("Final columns:")
print(df.columns.tolist())

# Check final shape
print("\nFinal shape:", df.shape)

Final columns:
['region', 'unemployment_rate', 'job_offers', 'skills_required', 'year', 'month', 'unemployment_level', 'market_tension', 'age_range', 'education_level', 'categoriec_offers', 'family_skills', 'categories_offers']

Final shape: (1000, 13)


In [30]:
# Display final dataset preview
print("\nFinal dataset preview:")
print(df.head(10))

# Display final types
print("\nFinal types:")
print(df.dtypes)


Final dataset preview:
         region  unemployment_rate  job_offers  \
0       Houston                6.8        4894   
1    Washington               11.1        2695   
2       Chicago                7.3        1174   
3  Indianapolis               11.2        3708   
4      New York               13.7         268   
5   Los Angeles                5.0        1785   
6       Phoenix               13.7        2784   
7      San Jose               10.1        4981   
8        Austin                9.0        2453   
9        Dallas               12.4         808   

                                     skills_required  year  month  \
0            Agile Methodologies, Project Management  2023     10   
1     Data Analysis, Ux/Ui Design, Digital Marketing  2025      5   
2  Agile Methodologies, Digital Marketing, Machin...  2024      9   
3                      Cloud Computing, Ux/Ui Design  2024     12   
4  Sql, Machine Learning, Python, Project Managem...  2023      9   
5  Cloud Co

In [31]:
# Convert to excel
df.to_excel("job_market_unemployment_trends.xlsx",index=False,sheet_name="job_market_unemployment_trends")

In [32]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [33]:
engine.connect()

In [34]:
df.to_sql(
    name="job_market_unemployment_trends",
    con=engine,
    if_exists="replace",
    index=False
)

1000